In [1]:
import pandas as pd
import torch
import os
import matplotlib.pyplot as plt
import numpy as np

In [2]:
DATA_DIR = "../data"

bars_seen_train         = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
bars_unseen_train       = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")
bars_seen_public_test   = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"), engine="fastparquet")
bars_seen_private_test  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

headlines_seen_train        = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_train.parquet"), engine="fastparquet")
headlines_unseen_train      = pd.read_parquet(os.path.join(DATA_DIR, "headlines_unseen_train.parquet"), engine="fastparquet")
headlines_seen_public_test  = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"), engine="fastparquet")
headlines_seen_private_test = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"), engine="fastparquet")

print("bars_seen_train:",         bars_seen_train.shape)
print("bars_unseen_train:",       bars_unseen_train.shape)
print("bars_seen_public_test:",   bars_seen_public_test.shape)
print("bars_seen_private_test:",  bars_seen_private_test.shape)
print("headlines_seen_train:",        headlines_seen_train.shape)
print("headlines_unseen_train:",      headlines_unseen_train.shape)
print("headlines_seen_public_test:",  headlines_seen_public_test.shape)
print("headlines_seen_private_test:", headlines_seen_private_test.shape)

bars_seen_train: (50000, 6)
bars_unseen_train: (50000, 6)
bars_seen_public_test: (500000, 6)
bars_seen_private_test: (500000, 6)
headlines_seen_train: (9740, 3)
headlines_unseen_train: (7631, 3)
headlines_seen_public_test: (99308, 3)
headlines_seen_private_test: (99148, 3)


In [3]:
def build_company_session_logs(df):
    df['company'] = df['headline'].str.split().str[:2].str.join(' ')
    # Group by session and company, then aggregate headlines into a list
    grouped_list_df = df.groupby(['session', 'company'])['headline'].apply(list).reset_index()

    # Rename the column for clarity
    grouped_list_df.rename(columns={'headline': 'all_headlines'}, inplace=True)
    
    return grouped_list_df


In [4]:
headlines_per_company_session = build_company_session_logs(headlines_seen_train)

In [5]:
headlines_per_company_session.head(10)

,session,company,all_headlines
0,0,Calvis Sciences,[Calvis Sciences secures $650M contract with a...
1,0,Orevex Renewables,[Orevex Renewables secures $500M contract with...
2,0,Relvos Biosciences,[Relvos Biosciences opens new office in Southe...
3,0,Yorvov Pharmaceuticals,[Yorvov Pharmaceuticals secures $180M contract...
4,1,Halvav Brands,[Halvav Brands sees mixed results in wireless ...
5,1,Jorvis Fuels,[Jorvis Fuels reports record quarterly revenue...
6,1,Strynal Industries,[Strynal Industries reports 15% decline in ope...
7,1,Zrovex Industries,[Zrovex Industries loses key contract in Asia ...
8,2,Kelvik Power,[Kelvik Power secures $800M contract with a le...
9,2,Zelval Energy,[Zelval Energy to host investor day focused on...


In [6]:
# Print the first 10 companies and their event sequences
print("--- Sample of Event Sequences ---")

for index, row in headlines_per_company_session.head(10).iterrows():
    # .ljust(25) adds spaces so the arrows all line up perfectly
    company_name = row['company'].ljust(25)
    sequence = row['all_headlines']
    
    print(f"Session {row['session']} | {company_name} -> {sequence}")

--- Sample of Event Sequences ---
Session 0 | Calvis Sciences           -> ['Calvis Sciences secures $650M contract with a leading cloud platform', 'Calvis Sciences expands operations into Asia Pacific markets']
Session 0 | Orevex Renewables         -> ['Orevex Renewables secures $500M contract with a global retailer', 'Orevex Renewables secures $650M contract with a major logistics provider', 'Orevex Renewables faces class action over automated logistics service disruption']
Session 0 | Relvos Biosciences        -> ['Relvos Biosciences opens new office in Southeast Asia', 'Relvos Biosciences names new head of precision manufacturing division', 'Relvos Biosciences reports 3% decline in operating income', 'Relvos Biosciences secures $320M contract with a leading cloud platform']
Session 0 | Yorvov Pharmaceuticals    -> ['Yorvov Pharmaceuticals secures $180M contract with a national infrastructure agency', 'Yorvov Pharmaceuticals sees mixed results in supply chain optimization pilot prog

In [7]:
def replace_headlines_with_keywords(headlines):
    new_headlines = []
    magic_words = [ "secures",
                    "expands",
                    "decline in operating income",
                    "improvement",
                    "opens",
                    "class action",
                    "loses key contract",
                    "withdraws",
                    "launches",
                    "recalls",
                    "misses",
                    "buyback program",
                    "droped",
                    "rising costs",
                    "mixed results",
                    "drop",
                    "breakthrough",
                    "acquisition",
                    "new head",
                    "strong demand",
                    "capital expenditure",
                    "scheduled maintenance",
                    "disruptions",
                    "excellence",
                    "facility upgrade",
                    "record quarterly revenue",
                    "explores strategic alternatives",
                    "routine patent applications",
                    "regulatory review",
                    "delays product launch",
                    "restructuring",
                    "revises long-term strategy",
                    "regulatory milestone",
                    "regulatory approval",
                    "host investor day",
                    "addresses investor concerns",
                    "confirms participation",
                    "enters joint venture",
                    "signs multi-year partnership",
                    "steps down unexpectedly",
                    "present at",
                    "lowers full-year guidance",
                    "publishes annual",
                    "schedules annual shareholder",
                    "major strategic initiative",
                    "potential merger",
                    "beats analyst expectations",
                    "raises full-year guidance",
                    "reports unexpected decline in"]
    
    ambiguous_headlines = []
    for headline in headlines:
        matches = 0
        for word in magic_words:
            if word in headline.lower():
                new_headlines.append(word)
                matches += 1
        if matches == 0:
            new_headlines.append(headline)    
        if matches > 1:
            ambiguous_headlines.append(headline)
            
    if len(ambiguous_headlines) != 0:
        print(f"Ambigous headlines:{ambiguous_headlines}")
    
    return new_headlines


In [8]:
headlines_per_company_session['encoded_keywords'] = headlines_per_company_session['all_headlines'].apply(replace_headlines_with_keywords)

In [9]:
master_long_headlines = []

# Assuming the column with the lists of text is named 'all_headlines'
for headline_list in headlines_per_company_session['encoded_keywords']:
    for text in headline_list:
        # Check if the text has more than 1 word
        if len(str(text).split()) > 4:
            master_long_headlines.append(text)

print(len(master_long_headlines))
for long_head in master_long_headlines:
    print(long_head) # Preview the first 5

0


In [10]:
unique_tuples = headlines_per_company_session['encoded_keywords'].apply(tuple).unique()

# 2. Convert them back to lists for easy reading
all_unique_patterns = [list(pattern) for pattern in unique_tuples]

print(f"Total unique patterns found: {len(all_unique_patterns)}\n")

# Print them all out
for pattern in all_unique_patterns:
    print(pattern)

Total unique patterns found: 2458

['secures', 'expands']
['secures', 'secures', 'class action']
['opens', 'new head', 'decline in operating income', 'secures']
['secures', 'mixed results']
['mixed results', 'rising costs']
['record quarterly revenue', 'secures', 'improvement', 'secures', 'new head', 'recalls']
['decline in operating income', 'acquisition']
['loses key contract', 'strong demand', 'acquisition', 'class action', 'rising costs', 'drop']
['secures', 'withdraws', 'secures', 'launches', 'recalls', 'misses']
['host investor day', 'misses']
['withdraws', 'scheduled maintenance']
['explores strategic alternatives', 'secures', 'strong demand']
['facility upgrade', 'steps down unexpectedly', 'routine patent applications', 'loses key contract']
['drop', 'revises long-term strategy']
['withdraws', 'acquisition']
['new head', 'buyback program']
['secures', 'launches']
['decline in operating income', 'buyback program', 'routine patent applications', 'record quarterly revenue']
['drop

In [11]:
frequency_df = headlines_per_company_session['encoded_keywords'].apply(tuple).value_counts().reset_index()
frequency_df.columns = ['encoded_keywords', 'frequency']

# 3. Convert the tuples back to lists for a cleaner look
frequency_df['encoded_keywords'] = frequency_df['encoded_keywords'].apply(list)

In [12]:
frequency_df[frequency_df["frequency"]>5]

,encoded_keywords,frequency
0,[secures],132
1,[acquisition],45
2,[revises long-term strategy],33
3,[scheduled maintenance],31
4,[mixed results],27
5,[regulatory review],24
6,[class action],23
7,[excellence],23
8,[delays product launch],23
9,"[secures, secures]",22


In [13]:
def print_nested_patterns(frequency_df):
    """
    Builds a Prefix Tree from the sequences and prints it hierarchically.
    Assumes frequency_df has columns: ['event_pattern', 'frequency']
    """
    # 1. Build the Tree
    tree = {}
    
    for _, row in frequency_df.iterrows():
        sequence = row['encoded_keywords']
        freq = row['frequency']
        
        # Skip empty sequences
        if not sequence:
            continue
            
        current_node = tree
        for event in sequence:
            if event not in current_node:
                # Initialize the node with a count and an empty dictionary for its children
                current_node[event] = {'_count': 0, 'children': {}}
            
            # Add the frequency to this specific path
            current_node[event]['_count'] += freq
            
            # Move down to the next level
            current_node = current_node[event]['children']

    # 2. Recursive function to print the tree with formatting
    def display_tree(node, prefix=""):
        # Sort the branches at this level by frequency (highest first)
        sorted_branches = sorted(node.items(), key=lambda item: item[1]['_count'], reverse=True)
        
        for i, (event, data) in enumerate(sorted_branches):
            is_last = (i == len(sorted_branches) - 1)
            count = data['_count']
            
            # Choose the correct pipe formatting
            branch_char = "└── " if is_last else "├── "
            
            # Print the current event and its frequency
            print(f"{prefix}{branch_char}[{event}] (Total: {count})")
            
            # Determine the prefix for the children of this event
            next_prefix = prefix + ("    " if is_last else "│   ")
            
            # Recursively print the children
            display_tree(data['children'], next_prefix)

    print("--- Nested Event Sequences ---\n")
    display_tree(tree)

# Execute the function using the frequency dataframe we created earlier
# print_nested_patterns(frequency_df)

In [14]:
print_nested_patterns(frequency_df)

--- Nested Event Sequences ---

├── [secures] (Total: 585)
│   ├── [secures] (Total: 70)
│   │   ├── [secures] (Total: 5)
│   │   │   ├── [decline in operating income] (Total: 1)
│   │   │   └── [regulatory review] (Total: 1)
│   │   ├── [withdraws] (Total: 4)
│   │   │   ├── [capital expenditure] (Total: 1)
│   │   │   │   └── [improvement] (Total: 1)
│   │   │   ├── [new head] (Total: 1)
│   │   │   └── [signs multi-year partnership] (Total: 1)
│   │   │       └── [rising costs] (Total: 1)
│   │   ├── [regulatory review] (Total: 3)
│   │   ├── [scheduled maintenance] (Total: 3)
│   │   │   ├── [decline in operating income] (Total: 1)
│   │   │   │   └── [launches] (Total: 1)
│   │   │   │       └── [new head] (Total: 1)
│   │   │   │           └── [secures] (Total: 1)
│   │   │   └── [breakthrough] (Total: 1)
│   │   ├── [buyback program] (Total: 3)
│   │   │   ├── [host investor day] (Total: 1)
│   │   │   │   └── [reports unexpected decline in] (Total: 1)
│   │   │   └── [new head]

In [15]:
def export_nested_patterns(frequency_df, filename="event_tree.txt"):
    # 1. Build the Tree (Same logic as before)
    tree = {}
    for _, row in frequency_df.iterrows():
        sequence = row['encoded_keywords']
        freq = row['frequency']
        if not sequence: continue
        current_node = tree
        for event in sequence:
            if event not in current_node:
                current_node[event] = {'_count': 0, 'children': {}}
            current_node[event]['_count'] += freq
            current_node = current_node[event]['children']

    # 2. Recursive function to WRITE to file instead of printing
    def write_tree(node, file, prefix=""):
        sorted_branches = sorted(node.items(), key=lambda item: item[1]['_count'], reverse=True)
        for i, (event, data) in enumerate(sorted_branches):
            is_last = (i == len(sorted_branches) - 1)
            count = data['_count']
            branch_char = "└── " if is_last else "├── "
            
            # Write to the file
            file.write(f"{prefix}{branch_char}[{event}] (Total: {count})\n")
            
            next_prefix = prefix + ("    " if is_last else "│   ")
            write_tree(data['children'], file, next_prefix)

    # Open the file and start writing
    with open(filename, 'w') as f:
        f.write("--- Nested Event Sequences ---\n\n")
        write_tree(tree, f)
    
    print(f"Success! Tree saved to {filename}. Open it in a text editor with Word Wrap disabled.")

# Run it:
# export_nested_patterns(frequency_df, "my_session_tree.txt")

In [16]:
export_nested_patterns(frequency_df, "headlines_tree_seen_test.txt")

Success! Tree saved to headlines_tree_seen_test.txt. Open it in a text editor with Word Wrap disabled.


In [17]:
DATA_DIR = "../data"

bars_seen_train         = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
bars_unseen_train       = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")
bars_seen_public_test   = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"), engine="fastparquet")
bars_seen_private_test  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

headlines_seen_train        = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_train.parquet"), engine="fastparquet")
headlines_unseen_train      = pd.read_parquet(os.path.join(DATA_DIR, "headlines_unseen_train.parquet"), engine="fastparquet")
headlines_seen_public_test  = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"), engine="fastparquet")
headlines_seen_private_test = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"), engine="fastparquet")

print("bars_seen_train:",         bars_seen_train.shape)
print("bars_unseen_train:",       bars_unseen_train.shape)
print("bars_seen_public_test:",   bars_seen_public_test.shape)
print("bars_seen_private_test:",  bars_seen_private_test.shape)
print("headlines_seen_train:",        headlines_seen_train.shape)
print("headlines_unseen_train:",      headlines_unseen_train.shape)
print("headlines_seen_public_test:",  headlines_seen_public_test.shape)
print("headlines_seen_private_test:", headlines_seen_private_test.shape)

bars_seen_train: (50000, 6)
bars_unseen_train: (50000, 6)
bars_seen_public_test: (500000, 6)
bars_seen_private_test: (500000, 6)
headlines_seen_train: (9740, 3)
headlines_unseen_train: (7631, 3)
headlines_seen_public_test: (99308, 3)
headlines_seen_private_test: (99148, 3)


In [18]:
combined_train  = pd.concat([headlines_seen_train, headlines_unseen_train]).sort_values(by=['session', 'bar_ix'])

In [19]:
combined_train.head(50)

,session,headline,bar_ix
0,0,Relvos Biosciences opens new office in Southea...,6
1,0,Orevex Renewables secures $500M contract with ...,12
2,0,Relvos Biosciences names new head of precision...,14
3,0,Calvis Sciences secures $650M contract with a ...,20
4,0,Yorvov Pharmaceuticals secures $180M contract ...,22
5,0,Relvos Biosciences reports 3% decline in opera...,26
6,0,Relvos Biosciences secures $320M contract with...,26
7,0,Calvis Sciences expands operations into Asia P...,33
8,0,Yorvov Pharmaceuticals sees mixed results in s...,44
9,0,Orevex Renewables secures $650M contract with ...,47


In [25]:
train_headlines_per_company_session = build_company_session_logs(combined_train)

In [26]:
train_headlines_per_company_session['encoded_keywords'] = train_headlines_per_company_session['all_headlines'].apply(replace_headlines_with_keywords)

In [27]:
unique_train_tuples = train_headlines_per_company_session['encoded_keywords'].apply(tuple).unique()

# 2. Convert them back to lists for easy reading
train_all_unique_patterns = [list(pattern) for pattern in unique_train_tuples]

print(f"Total unique patterns found: {len(all_unique_patterns)}\n")

# Print them all out
for pattern in train_all_unique_patterns:
    print(pattern)

Total unique patterns found: 2458

['secures', 'expands', 'new head', 'mixed results', 'scheduled maintenance']
['secures', 'secures', 'class action', 'scheduled maintenance', 'launches', 'secures']
['opens', 'new head', 'decline in operating income', 'secures', 'secures', 'present at']
['secures', 'mixed results', 'secures', 'improvement', 'class action']
['mixed results', 'rising costs', 'routine patent applications', 'new head']
['record quarterly revenue', 'secures', 'improvement', 'secures', 'new head', 'recalls', 'drop']
['decline in operating income', 'acquisition', 'breakthrough', 'secures', 'excellence', 'disruptions']
['loses key contract', 'strong demand', 'acquisition', 'class action', 'rising costs', 'drop', 'delays product launch', 'rising costs']
['secures', 'withdraws', 'secures', 'launches', 'recalls', 'misses', 'opens', 'class action']
['new head', 'breakthrough', 'improvement', 'expands']
['mixed results', 'secures', 'scheduled maintenance', 'drop', 'class action', '

In [23]:
train_frequency_df = train_headlines_per_company_session['encoded_keywords'].apply(tuple).value_counts().reset_index()
train_frequency_df.columns = ['encoded_keywords', 'frequency']

# 3. Convert the tuples back to lists for a cleaner look
train_frequency_df['encoded_keywords'] = train_frequency_df['encoded_keywords'].apply(list)

In [24]:
train_frequency_df[train_frequency_df["frequency"]>5]

,encoded_keywords,frequency
0,[secures],132
1,[acquisition],45
2,[revises long-term strategy],33
3,[scheduled maintenance],31
4,[mixed results],27
5,[regulatory review],24
6,[class action],23
7,[excellence],23
8,[delays product launch],23
9,"[secures, secures]",22
